# 61장 · FastMCP 도구 등록과 클라이언트 호출

MCP 서버는 호출할 수 있는 도구 목록과 각 도구의 입력 형식을 알립니다. 클라이언트는 도구 이름과 인자를 지정해 결과를 받습니다.

도서관 시간표 함수를 FastMCP 서버에 등록합니다. MCP 클라이언트에서 도구 명세를 조회하고 요일 인자를 전달해 서버의 실제 반환값을 읽습니다.

실행을 확인한 커널은 Python 3.12.15입니다. 예제에 사용한 외부 패키지는 mcp 1.30.0입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.


## FastMCP의 tool() 데코레이터

서버는 조회 함수를 소유합니다. 클라이언트는 서버의 목록에서 어떤 도구를 부를 수 있는지 확인합니다. 이 노트북은 MCP Python SDK의 메모리 전송을 사용합니다. 별도 프로세스를 띄우는 stdio와 외부 주소에 연결하는 HTTP는 장별 참고 구현에서 다룹니다.


### 셀 01 · FastMCP 서버와 도구 등록

수업용 MCP 1.x 커널에서 실행합니다. @server.tool()이 함수의 이름, 설명, day의 허용값을 도구 명세에 반영합니다.

직접 함수 호출에서는 반환한 운영 시간을 확인합니다. SDK 메모리 전송에서는 세션을 통해 받은 도구 결과를 읽습니다. 별도 프로세스로 서버를 띄우는 stdio는 또 다른 경로입니다. 메모리 전송의 성공만으로 stdio 연결까지 성공했다고 기록하지 않습니다.


In [1]:
from typing import Literal
from mcp.server.fastmcp import FastMCP
from mcp.shared.memory import create_connected_server_and_client_session

server = FastMCP("course-library-v1", log_level="ERROR")
hours = {"weekday": "09:00~18:00", "weekend": "10:00~16:00"}

@server.tool()
def get_library_hours(day: Literal["weekday", "weekend"]) -> dict[str, str]:
    """가상 도서관의 평일 또는 주말 운영 시간을 조회합니다."""
    return {"day": day, "hours": hours[day], "source": "수업용 가상 도서관"}
print(get_library_hours('weekend')['hours'])

10:00~16:00


10:00~16:00은 Python 함수를 직접 호출한 결과입니다. 여기까지는 아직 클라이언트가 서버에 요청한 적이 없습니다.


### 같은 함수의 기준 결과

<p>Python에서 get_library_hours('weekend')를 직접 부르면 10:00~16:00을 읽을 수 있습니다. 뒤의 세션 호출에서 같은 요일이 같은 값을 내는지 비교할 기준입니다.</p>


### 셀 02 · list_tools()로 명세 조회

연결 도우미가 세션을 열고 초기화합니다. 목록 조회에는 day를 보내지 않습니다.


In [2]:
async with create_connected_server_and_client_session(server) as session:
    catalog = await session.list_tools()
print(catalog.tools[0].name)

get_library_hours


셀 01의 직접 함수 호출과 달리 이 세션이 받은 것은 도구 목록입니다. 목록은 세션이 닫힌 뒤에도 catalog에 남습니다.

list_tools() 결과에 get_library_hours가 있다는 것은 클라이언트가 호출 명세를 읽었다는 뜻입니다. 목록 요청은 서버의 주말 시간표를 읽지 않습니다. 실제 연결 검증에는 이름 확인과 함께 인자를 보낸 뒤 반환값을 읽는 단계가 필요합니다.


#### 도구 이름을 발견한 뒤

<mark>list_tools()로 얻은 이름은 실행 결과를 포함하지 않습니다.</mark> 서버가 운영 시간을 제공하는 도구를 노출하는지 확인한 뒤, 요일을 넣어 call_tool()을 호출하세요. 목록 조회와 실제 질의 중 어디에서 실패했는지 나눠 볼 수 있습니다.

선수 개념 · 15장 Function Calling의 도구 명세와 호출 요청

### 도구를 제공하는 쪽과 호출하는 쪽

<p>서버에는 get_library_hours가 등록되어 있습니다. 클라이언트는 세션으로 도구의 명세를 조회하거나 인자를 담아 호출합니다. 이 노트북의 연결은 메모리 안에서 이루어집니다.</p>


### 목록 조회와 실제 호출

<p>list_tools()가 돌려주는 것은 호출 가능한 도구의 명세입니다. 운영 시간을 받으려면 새로 연 세션의 call_tool()에 이름과 day 값을 함께 보냅니다.</p>

<h3>메모리 세션에서 조회할 수 있는 두 결과</h3>

<p>셀 02의 세션에서 list_tools()가 반환한 첫 명세의 이름은 get_library_hours입니다. 셀 03에서는 새 메모리 세션을 열고 같은 이름과 day="weekend"를 call_tool에 보냅니다. 이번에는 도구 명세 대신 weekend와 10:00~16:00이 담긴 structuredContent를 읽습니다.</p>


## list_tools()와 call_tool()

call_tool은 도구 이름과 인자 사전을 보냅니다. await 뒤에는 서버가 보낸 결과가 돌아옵니다. 아래처럼 structuredContent를 읽으면 텍스트를 다시 파싱할 필요가 없습니다.


### 셀 03 · call_tool()의 구조화 결과

호출을 열린 세션 안에 두고, 결과가 어느 요일에 대한 것인지 확인하세요.


In [3]:
async with create_connected_server_and_client_session(server) as session:
    reply = await session.call_tool("get_library_hours", {"day": "weekend"})
print(reply.structuredContent["day"], reply.structuredContent["hours"])

weekend 10:00~16:00


weekend와 10:00~16:00이 함께 돌아옵니다. 입력의 요일이 결과의 요일과 맞는지 대조할 수 있습니다.


#### 세션을 통해 MCP 도구의 구조화 결과 받기

```mermaid
flowchart TD
nclient["클라이언트"]
  nsession["열린 세션"]
  nserver["등록한 도구"]
  nreply["구조화 결과"]
  nclient --> nsession
  nsession --> nserver
  nserver --> nreply
```

이 호출은 노트북 안의 메모리 연결을 사용합니다. 목록 조회 결과인 도구 이름과 실제 호출 결과를 비교하세요.

### 구조화된 필드에서 응답 읽기

<p>reply.structuredContent에서 day와 hours를 각각 읽습니다. 출력 첫 값이 weekend인지 먼저 보면, 다른 요일의 시간을 실수로 주말 응답으로 읽는 일을 피할 수 있습니다.</p>


### 과제 04 · 요일 인자를 바꿔 도구 호출

인자 하나를 바꾸어 평일 운영 시간을 읽으세요. 이름 목록 조회만으로 끝내지 마세요.


In [4]:
# 새 세션에서 weekday를 요청하고 hours를 출력하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>async with create_connected_server_and_client_session(server) as session:
    reply = await session.call_tool(&#x27;get_library_hours&#x27;, {&#x27;day&#x27;: &#x27;weekday&#x27;})
print(reply.structuredContent[&#x27;hours&#x27;])</code></pre>
<p>실행 결과</p><pre><code>09:00~18:00</code></pre>
<p>09:00~18:00입니다. 수업용 메모리 전송 안에서 클라이언트와 서버가 실제로 교환한 값입니다. 외부 도서관의 현재 운영 시간을 확인한 결과는 아닙니다. 별도 프로세스를 쓰는 경우는 클라이언트를 연결하기 전에 서버가 실행 중인지 확인하세요.</p>
</details>


## 복습과 실습 점검

### 과제 05 · 등록된 도구의 입력 계약 읽기

get_library_hours가 day에 어떤 값을 허용하는지 도구 목록에서 읽으세요. 함수 본문을 열 수 없는 클라이언트도 list_tools가 돌려준 스키마에서 허용값을 읽을 수 있습니다.


In [ ]:
# catalog에서 get_library_hours를 이름으로 찾아 day의 enum을 출력하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>hours_tool = next(item for item in catalog.tools if item.name == &quot;get_library_hours&quot;)
day_schema = hours_tool.inputSchema[&quot;properties&quot;][&quot;day&quot;]
print(day_schema[&quot;enum&quot;])</code></pre>
<p>실행 결과</p><pre><code>['weekday', 'weekend']</code></pre>
<p>weekday와 weekend는 서버 도구의 Literal 선언에서 온 선택지입니다. 목록 조회는 호출할 때 필요한 인자를 알려 줍니다. 어느 시간에 문을 여는지는 실제로 그 인자를 보내야 알 수 있습니다.</p>
</details>


## 별도 프로세스의 도구를 stdio로 호출하기

앞 셀은 서버와 클라이언트를 한 프로세스 안에 두었습니다. 여기서는 같은 시간표를 별도 Python 프로세스에서 제공합니다. 학생은 노트북 셀만 실행합니다. 커널이 서버를 실행하고, 세션을 닫으면 하위 프로세스도 함께 정리합니다.

### 셀 06 · 표준 입출력으로 요청을 보내고 결과 꺼내기

노트북에 서버 코드를 문자열로 준비합니다. 서버 프로세스의 표준 출력은 MCP 메시지를 전달하는 곳이므로 그 코드에는 디버그용 print를 넣지 않습니다. 초기화 후 주말 운영 시간을 물어보세요.

In [5]:
import sys
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

server_body = """from typing import Literal
from mcp.server.fastmcp import FastMCP
api = FastMCP("library-stdio", log_level="ERROR")

@api.tool(description="가상 도서관의 평일·주말 운영 시간을 조회합니다.")
def get_library_hours(day: Literal["weekday", "weekend"]) -> dict[str, str]:
    hours = {"weekday": "09:00~18:00", "weekend": "10:00~16:00"}
    return {"day": day, "hours": hours[day]}
"""

async def call_over_stdio(body, name, arguments):
    program = body + '\napi.run(transport="stdio")'
    params = StdioServerParameters(command=sys.executable, args=["-c", program])
    async with stdio_client(params) as (reader, writer):
        async with ClientSession(reader, writer) as client:
            await client.initialize()
            return await client.call_tool(name, arguments)

weekend_reply = await call_over_stdio(server_body, "get_library_hours", {"day": "weekend"})
print(weekend_reply.isError, weekend_reply.structuredContent["hours"])

False 10:00~16:00


False와 10:00~16:00가 보입니다. 클라이언트는 서버 프로세스 안의 hours 딕셔너리에 접근할 수 없습니다. call_tool에 이름과 직렬화할 수 있는 인자를 넘기고 MCP 응답의 필드에서 시간을 읽었습니다.

### 과제 07 · stdio 서버에 대출 일수 도구 추가하기

server_body 뒤에 @api.tool로 get_loan_days를 정의하세요. 인자는 없고 구조화 결과는 {"days": 14}입니다. 확장된 서버 코드로 새 프로세스를 시작해 그 도구를 호출하고 days를 출력하세요.

In [6]:
# 새 도구 정의를 server_body에 이어 붙이고 call_over_stdio로 실행하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>loan_server_body = server_body + &quot;&quot;&quot;
@api.tool(description=&quot;가상 도서관의 기본 대출 기간을 일 단위로 읽습니다.&quot;)
def get_loan_days() -&gt; dict[str, int]:
    return {&quot;days&quot;: 14}
&quot;&quot;&quot;
loan_reply = await call_over_stdio(loan_server_body, &quot;get_loan_days&quot;, {})
print(loan_reply.isError, loan_reply.structuredContent[&quot;days&quot;])</code></pre>
<p>실행 결과</p><pre><code>False 14</code></pre>
<p>False와 14가 출력되면 새 함수가 서버 쪽에서 등록되고 클라이언트가 결과를 받았습니다. 노트북 셀에 함수만 정의하고 server_body에 보내지 않으면 별도 프로세스는 그 함수를 모릅니다.</p>
</details>

<mark>서버에 등록할 함수는 실제로 실행되는 서버 프로세스 안에 있어야 합니다.</mark> 노트북에서 같은 이름의 함수를 직접 호출하는 실험으로는 MCP 등록이나 stdio 연결이 동작하는지 확인할 수 없습니다.

62장에서는 도구 이름과 인자를 유지하고 두 프로그램이 메시지를 전하는 방법을 바꿉니다.